# 09 — Hybrid: full-context linear-attn memory + short vanilla DT

A two-stream architecture for the cross-trial-memory problem:

- **Long stream** (`LongLinearMemory`) — causal linear-attention over the *entire* session. Emits a per-step memory vector at every state position. Cheap because linear attention is `O(L)`.
- **Short stream** (`DecisionTransformer`) — vanilla softmax DT with a small context (32-256 steps). The action head.

The long-stream hidden at step `t` is **added to the short DT's state input at step `t`** ("suggestions transposed onto the main DT"). The short DT then attends causally over the short window as usual; the additive bias lets it condition on cross-trial structure the short window can't see directly.

Trained end-to-end: each step picks a small batch of sessions, encodes the full session through the long stream once, samples a handful of short windows per session, sums long-stream memory into the short window's state inputs, and backprops a single cross-entropy loss through both streams.


## 0. Colab bootstrap

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryangrg/corner-maze-rl/blob/main/notebooks/09_hybrid_fullctx_linearattn_shortvanilladt.ipynb)

**Getting started on Colab:**

1. **Click the badge above** to open this notebook in Colab.
2. **Pick a GPU runtime** (training notebooks only): `Runtime → Change runtime type → T4` (free) or `L4` / `A100` (Pro). Required to get the bf16 / `torch.compile` / TF32 speedups; exploration notebooks (01, 02) are fine on CPU.
3. **Drive setup** (one-time, optional but recommended): in My Drive, create a folder called `corner-maze-rl-colab/`. The bootstrap auto-creates `runs/` inside it. Skip this and run artifacts stay ephemeral on the Colab VM disk — download before the runtime disconnects.
4. **Sign into Drive** when prompted by the first run of the bootstrap cell (one click).
5. **Run all** (`Runtime → Run all`). The cell below handles repo clone, package install, Drive mount, and starts a background sync. Nothing else to configure.

---

**How the bootstrap works** (no-op locally; on Colab):

- Clones this repo to `/content/corner-maze-rl` and installs it with `pip install -e .` (deps come from `pyproject.toml`).
- Training writes always go to `data/runs/` on the **Colab VM disk** — a real directory, not a symlink.
- A background daemon thread rsyncs `data/runs/` → `corner-maze-rl-colab/runs/` on Drive every `SYNC_INTERVAL_SEC` seconds (default 120s), plus a final sync on kernel exit.
- If Drive disconnects mid-run, training keeps working. The next sync attempt fails silently and the one after retries. Worst case you lose the last `SYNC_INTERVAL_SEC` seconds of artifacts.

Data inputs (lookups + yoked dataset) come with the cloned repo — no upload required.

**Manual sync any time**: call `sync_runs_to_drive(verbose=True)` from any cell.

In [ ]:
# ── Colab bootstrap (no-op locally; local-write + periodic Drive sync) ─
import sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    import os, subprocess, threading, atexit
    from pathlib import Path

    # ── Customise for your setup ──────────────────────────────────────
    REPO_URL          = 'https://github.com/ryangrg/corner-maze-rl.git'
    REPO_BRANCH       = 'main'
    REPO_DIR          = Path('/content/corner-maze-rl')
    USE_DRIVE         = True   # set False to skip Drive entirely
    DRIVE_ROOT        = Path('/content/drive/MyDrive/corner-maze-rl-colab')
    SYNC_INTERVAL_SEC = 120    # background rsync cadence
    # ──────────────────────────────────────────────────────────────────

    # 1. Clone repo (brings lookups + yoked dataset; both are committed)
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', '--depth=1',
                               '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        print(f'  repo already at {REPO_DIR}')

    # 2. Install package
    subprocess.check_call(['pip', 'install', '-q', '-e', str(REPO_DIR)])

    # 3. data/runs is ALWAYS a real local dir on the VM disk — never a symlink.
    #    This keeps writes resilient to mid-run Drive disconnects.
    local_runs = REPO_DIR / 'data' / 'runs'
    local_runs.mkdir(parents=True, exist_ok=True)

    # 4. Try Drive mount; failure just disables sync (training still works).
    drive_runs   = None
    drive_status = 'skipped (USE_DRIVE=False)'
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            if not DRIVE_ROOT.exists():
                drive_status = f'mounted, but {DRIVE_ROOT} missing — sync disabled'
            else:
                drive_runs = DRIVE_ROOT / 'runs'
                drive_runs.mkdir(exist_ok=True)
                drive_status = f'mounted at {DRIVE_ROOT}'
        except Exception as e:
            drive_status = f'mount failed ({type(e).__name__}) — sync disabled'

    # 5. Public sync helper — call from any cell with verbose=True for feedback.
    _sync_lock = threading.Lock()

    def sync_runs_to_drive(verbose: bool = False) -> tuple[bool, str]:
        """rsync local data/runs/ → Drive corner-maze-rl-colab/runs/.

        Returns (ok, message). Skips silently if Drive isn't reachable.
        Safe to call concurrently — uses a lock to prevent overlapping rsyncs.
        """
        if drive_runs is None:
            return (False, 'no drive target')
        if not _sync_lock.acquire(blocking=False):
            return (False, 'sync already in flight')
        try:
            result = subprocess.run(
                ['rsync', '-a', '--update',
                 str(local_runs) + '/', str(drive_runs) + '/'],
                capture_output=True, text=True, timeout=300,
            )
            ok  = (result.returncode == 0)
            msg = 'synced' if ok else f'rc={result.returncode}: {result.stderr.strip()[:200]}'
            if verbose:
                print(f'  [drive sync] {msg}')
            return (ok, msg)
        except Exception as e:
            if verbose:
                print(f'  [drive sync] {type(e).__name__}: {e}')
            return (False, f'{type(e).__name__}: {e}')
        finally:
            _sync_lock.release()

    # 6. Background sync — daemon thread, silent. Dies with the kernel.
    _sync_stop = threading.Event()
    def _sync_loop():
        while not _sync_stop.wait(SYNC_INTERVAL_SEC):
            sync_runs_to_drive(verbose=False)

    if drive_runs is not None:
        threading.Thread(target=_sync_loop, daemon=True).start()
        # Final sync on kernel exit so the last cycle's writes catch up.
        atexit.register(lambda: (_sync_stop.set(), sync_runs_to_drive(verbose=True)))
        sync_status = f'active (every {SYNC_INTERVAL_SEC}s + final on exit)'
    else:
        sync_status = 'inactive (runs are EPHEMERAL — download before disconnect)'

    # 7. cd into notebooks/ so REPO_ROOT = Path.cwd().parent resolves
    os.chdir(REPO_DIR / 'notebooks')

    print(f'\n✓ Colab bootstrap complete')
    print(f'  repo:    {REPO_DIR}  (branch={REPO_BRANCH})')
    print(f'  cwd:     {Path.cwd()}')
    print(f'  drive:   {drive_status}')
    print(f'  runs:    {local_runs}  (always local; never a symlink)')
    print(f'  sync:    {sync_status}')
    print(f'\n  manual sync any time: sync_runs_to_drive(verbose=True)')
else:
    print('✓ local environment — bootstrap skipped')


## 1. Setup


In [ ]:
from pathlib import Path
import json, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

import sys; sys.path.insert(0, '../scripts')
import train_dt as T

from corner_maze_rl.encoders.grid_cells import GridCellEncoder
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.models.decision_transformer import DTConfig, DecisionTransformer
from corner_maze_rl.models.linear_decision_transformer import LinearAttnEncoderLayer

DATA_PATH    = Path('../data/yoked/dataset/actions_synthetic_pretrial.parquet')
RUN_DIR      = Path('../runs/dt/nb09'); RUN_DIR.mkdir(parents=True, exist_ok=True)

FULL_CTX     = 4096     # long-stream cap (~99th percentile of session length)
SHORT_CTX    = 64       # short-stream context
EMBED_DIM    = 60
NUM_HEADS    = 4
LONG_LAYERS  = 2
SHORT_LAYERS = 2
LR           = 5e-4
WEIGHT_DECAY = 1e-4
SESSIONS_PER_STEP   = 4   # long-stream batch (full sessions are big)
WINDOWS_PER_SESSION = 8   # short-stream samples per session per step
EPOCHS       = 5          # full-context + hybrid is expensive — fewer epochs.
VAL_FRAC     = 0.10
SEED         = 0
MAX_SESSIONS = None

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device={device}  FULL_CTX={FULL_CTX}  SHORT_CTX={SHORT_CTX}')


## 2. Hybrid model


In [ ]:
class LongLinearMemory(nn.Module):
    """Causal linear-attn over the full session.

    Emits per-step hidden vectors at state-token positions, shape (B, L, D).
    Same I/O contract as LinearDecisionTransformer minus the action head.
    """
    def __init__(self, embed_dim, num_heads, num_layers,
                 num_actions=5, full_ctx=FULL_CTX,
                 dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.embed_rtg    = nn.Linear(1, embed_dim)
        self.embed_state  = nn.Linear(embed_dim, embed_dim)
        self.embed_action = nn.Linear(num_actions, embed_dim)
        self.position_emb = nn.Embedding(full_ctx, embed_dim)
        self.layers = nn.ModuleList([
            LinearAttnEncoderLayer(embed_dim, num_heads, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, rtg, state, action):
        b, k, _ = state.shape
        r = self.embed_rtg(rtg); s = self.embed_state(state); a = self.embed_action(action)
        p = self.position_emb(torch.arange(k, device=rtg.device)).unsqueeze(0)
        r, s, a = r + p, s + p, a + p
        x = torch.stack((r, s, a), dim=2).reshape(b, 3 * k, self.embed_dim)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return x[:, 1::3, :]   # state-position hiddens: (B, L, D)


class HybridDT(nn.Module):
    """Long-stream memory + short-stream vanilla DT.

    The long stream emits a memory vector at every timestep; the short DT
    receives the rat's pose state PLUS the long-stream memory at that step,
    added before the short DT's own state embedding.
    """
    def __init__(self, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 long_layers=LONG_LAYERS, short_layers=SHORT_LAYERS,
                 full_ctx=FULL_CTX, short_ctx=SHORT_CTX, num_actions=5):
        super().__init__()
        self.long = LongLinearMemory(embed_dim, num_heads, long_layers,
                                     num_actions=num_actions, full_ctx=full_ctx)
        short_cfg = DTConfig(embed_dim=embed_dim, num_actions=num_actions,
                             context_size=short_ctx, num_heads=num_heads,
                             num_layers=short_layers, pos_encoding='learned')
        self.short = DecisionTransformer(short_cfg)
        self.embed_dim = embed_dim
        self.short_ctx = short_ctx

    def long_encode(self, rtg_full, state_full, action_full):
        return self.long(rtg_full, state_full, action_full)

    def short_forward(self, rtg_win, state_win, action_win, mem_win):
        return self.short(rtg_win, state_win + mem_win, action_win)


## 3. Load + pack each session


In [ ]:
df = pd.read_parquet(DATA_PATH)
if MAX_SESSIONS is not None:
    keep = sorted(df['session_id'].unique())[:MAX_SESSIONS]
    df = df[df['session_id'].isin(keep)].reset_index(drop=True)

encoder = GridCellEncoder()
assert encoder.output_dim == EMBED_DIM

# 11*11*4 = 484 distinct poses — store pose IDs (int16) per step, expand to a
# 60-D vector on-GPU at batch time. Cuts dataset RAM by ~120x vs storing the
# materialized state vectors.
N_X, N_Y, N_D = 11, 11, 4
N_POSES = N_X * N_Y * N_D
pose_table_np = np.zeros((N_POSES, EMBED_DIM), dtype=np.float32)
for x in range(1, N_X + 1):
    for y in range(1, N_Y + 1):
        for d in range(N_D):
            pose_table_np[((x - 1) * N_Y + (y - 1)) * N_D + d] = encoder.encode(x, y, d)
pose_table = torch.from_numpy(pose_table_np).to(device)

def pack_full_ids(sdf, full_ctx):
    if len(sdf) > full_ctx:
        sdf = sdf.iloc[-full_ctx:]
    n = len(sdf)
    xs = sdf['grid_x'].to_numpy(np.int32)
    ys = sdf['grid_y'].to_numpy(np.int32)
    ds = sdf['direction'].to_numpy(np.int32)
    pose_orig = (((xs - 1) * N_Y + (ys - 1)) * N_D + ds).astype(np.int16)
    rewarded = sdf['rewarded'].to_numpy(np.float32)
    rtg_orig = np.flip(np.cumsum(np.flip(rewarded))).astype(np.float32)
    action_orig = sdf['action'].to_numpy(np.int64).clip(0, 4).astype(np.int8)
    pose_ids = np.zeros(full_ctx, dtype=np.int16);   pose_ids[:n]   = pose_orig
    action_ids = np.zeros(full_ctx, dtype=np.int8);  action_ids[:n] = action_orig
    rtg = np.zeros(full_ctx, dtype=np.float32);      rtg[:n]        = rtg_orig
    return pose_ids, action_ids, rtg, n

sids = df['session_id'].unique().tolist()
sessions = []
for sid in sids:
    sdf = df[df['session_id'] == sid].sort_values('step')
    p, a, r, n = pack_full_ids(sdf, FULL_CTX)
    sessions.append({'sid': sid, 'pose_ids': p, 'action_ids': a, 'rtg': r, 'n': n})

ram_mb = sum(s['pose_ids'].nbytes + s['action_ids'].nbytes + s['rtg'].nbytes
             for s in sessions) / 1e6
print(f'sessions packed: {len(sessions)}   dataset RAM ≈ {ram_mb:.0f} MB')

rng = np.random.default_rng(SEED)
order = rng.permutation(len(sessions))
n_val = max(1, int(len(sessions) * VAL_FRAC))
val_idx   = order[:n_val].tolist()
train_idx = order[n_val:].tolist()
print(f'sessions: train={len(train_idx)}  val={len(val_idx)}   FULL_CTX={FULL_CTX}')


## 4. Train (joint, end-to-end)


In [ ]:
from tqdm.auto import tqdm

model = HybridDT().to(device)
n_params = sum(p.numel() for p in model.parameters())
model = torch.compile(model)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ce = nn.CrossEntropyLoss()

USE_BF16 = (device == 'cuda')
print(f'params={n_params:,}  bf16={USE_BF16}  compiled=True')

def stack_batch(indices):
    poses = np.stack([sessions[i]['pose_ids']   for i in indices])
    acts  = np.stack([sessions[i]['action_ids'] for i in indices])
    rtgs  = np.stack([sessions[i]['rtg']        for i in indices])
    pose_ids   = torch.from_numpy(poses).to(device, non_blocking=True).long()
    action_ids = torch.from_numpy(acts).to(device, non_blocking=True).long()
    rtg        = torch.from_numpy(rtgs).to(device, non_blocking=True).unsqueeze(-1)
    state  = pose_table[pose_ids]                                              # (B, L, 60)
    action = nn.functional.one_hot(action_ids, num_classes=5).float()          # (B, L, 5)
    ns = np.array([sessions[i]['n'] for i in indices], dtype=np.int64)
    return rtg, state, action, ns

def sample_windows(ns, m, rng):
    """For each session length n_i, sample m random window-end positions in [SHORT_CTX-1, n_i)."""
    out = np.empty((len(ns), m), dtype=np.int64)
    for i, n in enumerate(ns):
        hi = max(SHORT_CTX, int(n))
        out[i] = rng.integers(low=SHORT_CTX - 1, high=hi, size=m)
    return torch.from_numpy(out)

def gather_windows(full, end_idx):
    """full: (B, L_full, D)   end_idx: (B, M)   -> (B*M, SHORT_CTX, D)."""
    B, L, D = full.shape
    M = end_idx.shape[1]
    offs = torch.arange(SHORT_CTX, device=full.device).view(1, 1, SHORT_CTX)
    starts = (end_idx.to(full.device) - (SHORT_CTX - 1)).unsqueeze(-1)
    idx = (starts + offs).clamp(min=0)
    idx_exp = idx.unsqueeze(-1).expand(B, M, SHORT_CTX, D)
    full_exp = full.unsqueeze(1).expand(B, M, L, D)
    return torch.gather(full_exp, dim=2, index=idx_exp).reshape(B * M, SHORT_CTX, D)

def run_epoch(idx_list, train: bool, rng, epoch_no: int):
    model.train() if train else model.eval()
    sum_loss = sum_correct = sum_tokens = 0.0
    ctx_mgr = torch.enable_grad() if train else torch.no_grad()
    idx_arr = np.array(idx_list)
    if train: rng.shuffle(idx_arr)
    steps = len(idx_arr) // SESSIONS_PER_STEP
    desc = f'ep {epoch_no:3d}/{EPOCHS} ' + ('train' if train else 'val')
    pbar = tqdm(range(steps), desc=desc, leave=False)
    with ctx_mgr:
        for st in pbar:
            batch_idx = idx_arr[st * SESSIONS_PER_STEP:(st + 1) * SESSIONS_PER_STEP].tolist()
            rtg, state, action, ns = stack_batch(batch_idx)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16):
                mem = model.long_encode(rtg, state, action)
                ends = sample_windows(ns, WINDOWS_PER_SESSION, rng)
                rtg_w    = gather_windows(rtg,    ends)
                state_w  = gather_windows(state,  ends)
                action_w = gather_windows(action, ends)
                mem_w    = gather_windows(mem,    ends)
                logits = model.short_forward(rtg_w, state_w, action_w, mem_w)
                targets = action_w.argmax(dim=-1)
                loss = ce(logits.reshape(-1, 5), targets.reshape(-1))
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            sum_loss    += loss.item() * targets.numel()
            sum_correct += (logits.argmax(-1) == targets).sum().item()
            sum_tokens  += targets.numel()
            if train and sum_tokens > 0:
                pbar.set_postfix(loss=f'{sum_loss/sum_tokens:.4f}',
                                 acc=f'{sum_correct/sum_tokens:.3f}')
    return sum_loss / max(sum_tokens, 1), sum_correct / max(sum_tokens, 1)

history = []
t0 = time.time()
rng = np.random.default_rng(SEED)
for epoch in range(1, EPOCHS + 1):
    ep0 = time.time()
    train_loss, train_acc = run_epoch(train_idx, train=True,  rng=rng, epoch_no=epoch)
    val_loss,   val_acc   = run_epoch(val_idx,   train=False, rng=rng, epoch_no=epoch)
    ep_sec = time.time() - ep0
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc, 'epoch_sec': ep_sec})
    print(f'[epoch {epoch:3d}] train_loss={train_loss:.4f} acc={train_acc:.3f} '
          f'| val_loss={val_loss:.4f} acc={val_acc:.3f}  '
          f'({ep_sec:.1f}s/epoch, {time.time()-t0:.0f}s total)')

ckpt = RUN_DIR / 'model.pt'
state_dict = (model._orig_mod if hasattr(model, '_orig_mod') else model).state_dict()
torch.save({'state_dict': state_dict, 'arch': 'hybrid',
            'short_ctx': SHORT_CTX, 'full_ctx': FULL_CTX,
            'embed_dim': EMBED_DIM, 'num_heads': NUM_HEADS,
            'long_layers': LONG_LAYERS, 'short_layers': SHORT_LAYERS}, ckpt)
(RUN_DIR / 'metrics.jsonl').write_text(''.join(json.dumps(h)+'\n' for h in history))
print(f'\nsaved checkpoint -> {ckpt}')


## 5. Inference movie

Re-encode the growing rollout history through the long stream each step (`O(L)` linear attention; cheap), gather long-stream memory at the latest `SHORT_CTX` positions, sum it into the short DT's state inputs.


In [ ]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw

RTG_TARGET   = 20.0
MAX_STEPS    = 600
BASE_TEMP    = 1.0
ROLLOUT_SEED = 0
MOVIE_PATH   = RUN_DIR / 'rollout.mp4'
ACTION_NAMES = ['Left', 'Right', 'Forward', 'EnterWell', 'Pause']

model.eval()
env = CornerMazeEnv(session_type='exposure', obs_mode='view', render_mode='rgb_array')
env.reset(seed=ROLLOUT_SEED)

hist_state, hist_action, hist_rtg = [], [], []
frames = []
last_pose = None; stagnation = 0
total_reward = 0.0; n_well_rewards = 0

for step in range(MAX_STEPS):
    x, y, d = int(env.agent_pos[0]), int(env.agent_pos[1]), int(env.agent_dir)
    if (x, y, d) == last_pose: stagnation += 1
    else: stagnation = 0
    last_pose = (x, y, d)
    temp = BASE_TEMP + (stagnation // 3) * 1.5

    s_vec = encoder.encode(x, y, d).astype(np.float32)
    hist_state.append(s_vec)
    hist_rtg.append(np.array([RTG_TARGET], dtype=np.float32))
    if len(hist_action) < len(hist_state):
        hist_action.append(np.zeros(5, dtype=np.float32))

    L = len(hist_state)
    if L > FULL_CTX:
        hist_state  = hist_state[-FULL_CTX:]
        hist_action = hist_action[-FULL_CTX:]
        hist_rtg    = hist_rtg[-FULL_CTX:]
        L = FULL_CTX
    rtg_t    = torch.from_numpy(np.stack(hist_rtg)).unsqueeze(0).to(device)
    state_t  = torch.from_numpy(np.stack(hist_state)).unsqueeze(0).to(device)
    action_t = torch.from_numpy(np.stack(hist_action)).unsqueeze(0).to(device)

    with torch.no_grad():
        mem_full = model.long_encode(rtg_t, state_t, action_t)   # (1, L, D)
        start = max(0, L - SHORT_CTX)
        rtg_w    = rtg_t   [:, start:L]
        state_w  = state_t [:, start:L]
        action_w = action_t[:, start:L]
        mem_w    = mem_full[:, start:L]
        logits = model.short_forward(rtg_w, state_w, action_w, mem_w)
        probs = torch.softmax(logits[0, -1, :] / temp, dim=-1)
        action = int(torch.multinomial(probs, num_samples=1).item())

    img = env.render()
    canvas = Image.fromarray(img).resize((416, 416)).convert('RGB')
    panel = Image.new('RGB', (416, 464), 'black')
    panel.paste(canvas, (0, 48))
    draw = ImageDraw.Draw(panel)
    label = (f'step={step:3d} act={ACTION_NAMES[action]} '
             f'rtg={RTG_TARGET:.1f} ret={total_reward:+.2f} wells={n_well_rewards} L={L}')
    if stagnation > 0: label += f' stuck={stagnation} T={temp:.1f}'
    draw.text((6, 6),  label,                 fill='white')
    draw.text((6, 24), f'pose=({x},{y},{d})', fill='white')
    frames.append(np.array(panel))

    _, reward, term, trunc, _ = env.step(action)
    total_reward += reward
    if reward > 0.5: n_well_rewards += 1
    oh = np.zeros(5, dtype=np.float32); oh[action] = 1.0
    hist_action[-1] = oh
    if term or trunc:
        break

print(f'rollout: {len(frames)} steps, total_reward={total_reward:+.3f}, '
      f'wells={n_well_rewards}')
imageio.mimwrite(MOVIE_PATH, frames, fps=10, codec='libx264')
print(f'wrote -> {MOVIE_PATH}')


## 6. Watch the movie


In [ ]:
from IPython.display import Video
Video(str(MOVIE_PATH), embed=False, width=500)
